### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [24]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"]="true"

In [25]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EC0C567AA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EC0A4276B0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [26]:
from langchain.tools import tool

@tool
def get_weather(city:str)->str:
    """ get the city weather using this tool """
    return "the weather in {city} is sunny"

model_with_tool = model.bind_tools([get_weather])

In [27]:
response = model_with_tool.invoke("what is the weather in chennai, tamilnadu, india")
print(response)
response.tool_calls

content='' additional_kwargs={'reasoning_content': 'We need to get weather for Chennai. Use function get_weather.', 'tool_calls': [{'id': 'fc_254a341f-92a4-4bad-b416-850cc8157512', 'function': {'arguments': '{"city":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 134, 'total_tokens': 176, 'completion_time': 0.0939162, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.008367757, 'prompt_tokens_details': None, 'queue_time': 0.053268162, 'total_time': 0.102283957}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb1a62a098', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fef87-5b68-7e70-a9c9-bdcdb9313785-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Chennai'}, 'id': 'fc_254a341f-92a4-4bad-b416-850cc8157512', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 1

[{'name': 'get_weather',
  'args': {'city': 'Chennai'},
  'id': 'fc_254a341f-92a4-4bad-b416-850cc8157512',
  'type': 'tool_call'}]

### Tool Execution Loops

In [28]:
messages = [{"role":"user", "content":"what is the weather in chennai"}]
response = model_with_tool.invoke(messages)
messages.append(response)
response



AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks weather in Chennai. Use get_weather function.', 'tool_calls': [{'id': 'fc_ccfaa42f-6419-4167-a2b1-72b45315c04c', 'function': {'arguments': '{"city":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 128, 'total_tokens': 168, 'completion_time': 0.08475797, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.004921409, 'prompt_tokens_details': None, 'queue_time': 0.162124425, 'total_time': 0.089679379}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fef87-5c7f-7e83-bc8e-c846dbf3d1f3-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Chennai'}, 'id': 'fc_ccfaa42f-6419-4167-a2b1-72b45315c04c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input

In [29]:
for tool_call in response.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

messages

[{'role': 'user', 'content': 'what is the weather in chennai'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks weather in Chennai. Use get_weather function.', 'tool_calls': [{'id': 'fc_ccfaa42f-6419-4167-a2b1-72b45315c04c', 'function': {'arguments': '{"city":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 128, 'total_tokens': 168, 'completion_time': 0.08475797, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.004921409, 'prompt_tokens_details': None, 'queue_time': 0.162124425, 'total_time': 0.089679379}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fef87-5c7f-7e83-bc8e-c846dbf3d1f3-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Chennai'}, 'id': 'fc_ccfaa42f-6419-4167-a2b1-72b45315c04c', 'ty

In [30]:
final_response = model_with_tool.invoke(messages)
final_response.text

'The weather in Chennai is sunny.'